# SefariaExport על Kaggle

- הפעל Internet למחברת.
- הרץ את תאי הקוד מלמעלה למטה.
- כברירת מחדל המחברת מקבעת את `SefariaExport` ל-commit `be22bd00bee3cbe67fb2b99e7af88d162435a0e1`.
- MongoDB מותקן מבינרי Community רשמיים בגרסה `6.0.27`, תואם ל-Compose.
- הקבצים הסופיים מועתקים ל-`/kaggle/working`.
- העלאת Release ל-GitHub אופציונלית ומכובה כברירת מחדל.


In [ ]:
%%bash
set -euo pipefail

BASE="/tmp/sefariaexport_run"
OUT_DIR="/kaggle/working"
CONFIG="$BASE/notebook.env"
LOG_DIR="$BASE/logs"

rm -rf "$BASE"
mkdir -p "$BASE" "$OUT_DIR" "$LOG_DIR"

cat > "$CONFIG" <<'EOF'
export BASE=/tmp/sefariaexport_run
export OUT_DIR=/kaggle/working
export LOG_DIR=/tmp/sefariaexport_run/logs
export TZ_NAME=Asia/Jerusalem
export PYTHON_VERSION=3.9
export MONGO_VERSION=6.0.27
export MONGO_TOOLS_VERSION=100.9.4
export SEFARIAEXPORT_REF=be22bd00bee3cbe67fb2b99e7af88d162435a0e1
export DUMP_TGZ_PATH=
export ENABLE_GITHUB_RELEASE=false
export GH_REPO=
EOF

printf 'Configured: %s\n' "$CONFIG"
printf 'Output dir: %s\n' "$OUT_DIR"
printf 'Edit %s if you want a custom dump or GitHub release upload.\n' "$CONFIG"


In [ ]:
%%bash
set -euo pipefail
source /tmp/sefariaexport_run/notebook.env

timestamp() { date '+%H:%M:%S'; }
log() { printf '\n[%s] %s\n' "$(timestamp)" "$1"; }
run_step() {
  local name="$1"
  shift
  local log_file="$LOG_DIR/${name}.log"
  if "$@" >"$log_file" 2>&1; then
    printf '  ok\n'
  else
    printf '  failed, tail of %s:\n' "$log_file"
    tail -n 80 "$log_file"
    return 1
  fi
}
cleanup() {
  if pgrep -f "mongod.*$BASE/mongo-data" >/dev/null 2>&1; then
    pkill -f "mongod.*$BASE/mongo-data" || true
  fi
}
trap cleanup EXIT

BIN_DIR="/tmp/bin"
MM_ROOT="/tmp/micromamba"
MM="$BIN_DIR/micromamba"
mkdir -p "$BIN_DIR" "$MM_ROOT" "$LOG_DIR" "$BASE/bin" "$BASE/mongo-data" "$BASE/mongo-logs"

if [ ! -x "$MM" ]; then
  log "Installing micromamba"
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C "$BIN_DIR" --strip-components=1 bin/micromamba
fi

if ! "$MM" -r "$MM_ROOT" env list | awk '{print $1}' | grep -qx 'sefariaexport'; then
  log "Creating Python ${PYTHON_VERSION} environment"
  run_step "create_python_env" "$MM" -r "$MM_ROOT" create -y -n sefariaexport -c conda-forge "python=${PYTHON_VERSION}" pip git curl wget tar unzip jq
fi

log "Updating pip"
run_step "upgrade_pip" "$MM" -r "$MM_ROOT" run -n sefariaexport pip install -U pip setuptools wheel

log "Cloning pinned SefariaExport"
cd "$BASE"
if [ ! -d "$BASE/SefariaExport/.git" ]; then
  run_step "clone_sefariaexport" git clone https://github.com/Otzaria/SefariaExport.git
else
  printf '  ok\n'
fi
cd "$BASE/SefariaExport"
run_step "checkout_sefariaexport_ref" git checkout --detach "$SEFARIAEXPORT_REF"

export GITHUB_WORKSPACE="$PWD"
export MONGO_HOST=127.0.0.1
export MONGO_PORT=27017
export MONGO_DB_NAME=sefaria
export RESTORE_INDEXES_MODE=skip_links
export SEFARIA_EXPORT_PATH="$GITHUB_WORKSPACE/exports"
export PATH="$BASE/bin:$PATH"
export PIP_NO_CACHE_DIR=1
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_ROOT_USER_ACTION=ignore

log "Computing timestamp"
if ! source ./01_compute_timestamp.sh >"$LOG_DIR/01_compute_timestamp.log" 2>&1; then
  tail -n 80 "$LOG_DIR/01_compute_timestamp.log"
  exit 1
fi
printf '  %s\n' "$TS_STAMP"

log "Installing base tools"
run_step "02_install_base_tools" bash 02_install_base_tools.sh

log "Installing MongoDB Database Tools"
run_step "install_mongo_tools" bash -lc '
  TGZ="mongodb-database-tools-ubuntu2204-x86_64-${MONGO_TOOLS_VERSION}.tgz"
  wget -q -O "$BASE/$TGZ" "https://fastdl.mongodb.org/tools/db/$TGZ"
  tar -xzf "$BASE/$TGZ" -C "$BASE"
  cp -a "$BASE/mongodb-database-tools-ubuntu2204-x86_64-${MONGO_TOOLS_VERSION}/bin/"* "$BASE/bin/"
  rm -rf "$BASE/mongodb-database-tools-ubuntu2204-x86_64-${MONGO_TOOLS_VERSION}" "$BASE/$TGZ"
'

log "Installing MongoDB ${MONGO_VERSION}"
run_step "install_mongod" bash -lc '
  TGZ="mongodb-linux-x86_64-ubuntu2204-${MONGO_VERSION}.tgz"
  curl -fsSL -o "$BASE/$TGZ" "https://fastdl.mongodb.org/linux/$TGZ"
  curl -fsSL -o "$BASE/$TGZ.sha256" "https://fastdl.mongodb.org/linux/$TGZ.sha256"
  (cd "$BASE" && sha256sum -c "$(basename "$TGZ").sha256")
  rm -rf "$BASE/mongo-server"
  mkdir -p "$BASE/mongo-server"
  tar -xzf "$BASE/$TGZ" -C "$BASE/mongo-server" --strip-components=1
  rm -f "$BASE/$TGZ" "$BASE/$TGZ.sha256"
'

log "Starting MongoDB"
run_step "start_mongod" bash -lc '
  rm -rf "$BASE/mongo-data"
  mkdir -p "$BASE/mongo-data" "$BASE/mongo-logs"
  nohup "$BASE/mongo-server/bin/mongod" \
    --dbpath "$BASE/mongo-data" \
    --bind_ip 127.0.0.1 \
    --port 27017 \
    --wiredTigerCacheSizeGB 2.5 \
    --logpath "$BASE/mongo-logs/mongod.log" \
    --logappend \
    > "$BASE/mongo-logs/nohup.out" 2>&1 &
'
run_step "11_wait_for_mongodb" bash 11_wait_for_mongodb.sh

if [ -n "$DUMP_TGZ_PATH" ]; then
  log "Preparing custom dump"
  run_step "prepare_custom_dump" bash -lc '
    rm -rf mongo_dump_pkg mongo_dump dump_small.tar.gz
    mkdir -p mongo_dump_pkg mongo_dump
    cp -a "$DUMP_TGZ_PATH" dump_small.tar.gz
    tar -xzf dump_small.tar.gz -C mongo_dump
    rm -f dump_small.tar.gz
    if [ -d mongo_dump/dump/sefaria ]; then
      cp -a mongo_dump/dump/sefaria mongo_dump_pkg/
    elif [ -d mongo_dump/sefaria ]; then
      cp -a mongo_dump/sefaria mongo_dump_pkg/
    else
      echo "sefaria folder not found in dump"
      exit 1
    fi
    rm -rf mongo_dump
  '
else
  log "Downloading default dump"
  run_step "04_download_small_dump" bash 04_download_small_dump.sh
fi

log "Cloning Sefaria-Project"
run_step "05_clone_sefaria_project" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 05_clone_sefaria_project.sh

log "Installing build dependencies"
if ! run_step "06_install_build_deps" bash 06_install_build_deps.sh; then
  printf '  continuing without extra build deps\n'
fi

log "Installing Python requirements"
if ! run_step "07_pip_install_requirements" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 07_pip_install_requirements.sh; then
  log "Using google-re2 fallback"
  run_step "08_fallback_built_google_re2" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 08_fallback_built_google_re2.sh
fi

rm -rf "$SEFARIA_EXPORT_PATH"
mkdir -p "$SEFARIA_EXPORT_PATH"

log "Preparing export settings"
run_step "09_create_exports_dir" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 09_create_exports_dir.sh
run_step "10_create_local_settings" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 10_create_local_settings.sh

log "Restoring MongoDB dump"
run_step "12_restore_db_from_dump" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 12_restore_db_from_dump.sh

log "Running exports"
run_step "13_check_export_module" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 13_check_export_module.sh
run_step "14_run_exports" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 14_run_exports.sh
run_step "15_verify_exports" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 15_verify_exports.sh
if ! run_step "16_drop_db" bash 16_drop_db.sh; then
  printf '  continuing after non-blocking drop-db step\n'
fi

log "Post-processing and archiving"
run_step "17a_remove_english_in_exports" bash 17a_remove_english_in_exports.sh
run_step "17b_flatten_hebrew_in_exports" bash 17b_flatten_hebrew_in_exports.sh
run_step "17_build_combined_archive" bash 17_build_combined_archive.sh
run_step "18_split_archive" bash 18_split_archive.sh

shopt -s nullglob
ARTIFACTS=( "sefaria-exports-${TS_STAMP}.tar.zst" "sefaria-exports-${TS_STAMP}.tar.zst.part-"* )
FINAL_ARTIFACTS=()
for f in "${ARTIFACTS[@]}"; do
  if [ -e "$f" ]; then
    FINAL_ARTIFACTS+=("$f")
  fi
done
if [ "${#FINAL_ARTIFACTS[@]}" -eq 0 ]; then
  echo "❌ no archive files were created"
  exit 1
fi
cp -f "${FINAL_ARTIFACTS[@]}" "$OUT_DIR/"
(cd "$OUT_DIR" && sha256sum "${FINAL_ARTIFACTS[@]##*/}" > "sefaria-exports-${TS_STAMP}.sha256")
printf '%s\n' "$TS_STAMP" > "$OUT_DIR/sefaria-exports-${TS_STAMP}.stamp.txt"

if [ "$ENABLE_GITHUB_RELEASE" = "true" ]; then
  log "Publishing GitHub release"
  : "${GH_REPO:?Set GH_REPO in $CONFIG when ENABLE_GITHUB_RELEASE=true}"
  export GH_REPO
  export GH_TOKEN="$(python3 -c 'from kaggle_secrets import UserSecretsClient; print(UserSecretsClient().get_secret("GH_TOKEN"))')"
  run_step "19_ensure_gh_cli" bash 19_ensure_gh_cli.sh
  run_step "20_create_or_update_release" bash 20_create_or_update_release.sh
  run_step "21_upload_release_assets" bash 21_upload_release_assets.sh
fi

log "Done"
printf '  TS_STAMP=%s\n' "$TS_STAMP"
ls -lh "$OUT_DIR"/sefaria-exports-"$TS_STAMP"* "$OUT_DIR"/sefaria-exports-"$TS_STAMP".sha256
printf '  logs: %s\n' "$LOG_DIR"
